In [1]:
import os
import numpy as np
import pandas as pd



In [2]:
#--------------------------------------------------
# Custom errors
#--------------------------------------------------
class MyError(Exception):

    """Base class for custom errors."""

class InvalidCityError(MyError):

    """Raised when city not defined/not part of dataset."""

class InvalidColumnTypeError(MyError):

    """Raised when column is not present or do not contain expected data type."""

class InvalidColumnCountError(MyError):

    """Raised when column is different than expected."""

#--------------------------------------------------
# Helper data
#--------------------------------------------------

In [7]:
COLUMN_COUNT = 6
COL_NAMES=['year','month','day','temp','temp2','city']
# Weather data location
weather_data = "/Users/chaitalichakraborty/Desktop/personal/programmingformalismscourse2026/programming_formalisms_project_summer_2026/data/uppsala_tm_1722-2022.dat"

In [22]:
# City dictionary
city_dict = {
    "Uppsala"           :   1,
    "Risinge"           :   2,
    "Betna"             :   3,
    "Linköping"         :   4,
    "Stockholm"         :   5,
    "Interpolated"      :   6,
}


In [9]:
def file_is_tsv(file):
    """Checks if file exists and is readable as TSV."""
    try:
        data = pd.read_fwf(file)
        return data
    except Exception as e:
        print(f"Invalid TSV file: {e}")
        return None


In [10]:
file_is_tsv(weather_data)

,1722,1,12,1.9,1.8,1.1
0,1722,1,13,2.3,2.2,1
1,1722,1,14,1.8,1.7,1
2,1722,1,15,0.9,0.8,1
3,1722,1,16,-1.8,-1.9,1
4,1722,1,17,0.5,0.4,1
...,...,...,...,...,...,...
109921,2022,2,27,0.1,-0.2,1
109922,2022,2,28,-4.1,-4.4,1
109923,2022,2,29,2.8,2.6,1
109924,2022,2,30,4.2,4.0,1


In [11]:
def load_data(path: str) -> pd.DataFrame:
    """Reads a TSV file and returns a dataframe."""
    return pd.read_fwf(path)

In [12]:
load_data(weather_data)

,1722,1,12,1.9,1.8,1.1
0,1722,1,13,2.3,2.2,1
1,1722,1,14,1.8,1.7,1
2,1722,1,15,0.9,0.8,1
3,1722,1,16,-1.8,-1.9,1
4,1722,1,17,0.5,0.4,1
...,...,...,...,...,...,...
109921,2022,2,27,0.1,-0.2,1
109922,2022,2,28,-4.1,-4.4,1
109923,2022,2,29,2.8,2.6,1
109924,2022,2,30,4.2,4.0,1


In [13]:
def weather_df(filename):
    """Validates weather data format."""
    
    data = file_is_tsv(filename)

    if data is None:
        return None, False

    # Check number of columns
    if data.shape[1] != 6:
        print("Weather file must have 6 columns")
        return None, False

    # Expected types
    expected_types = [
        "int",
        "int",
        "int",
        "float",
        "float",
        "int"
    ]

    for i, expected in enumerate(expected_types):
        col = data.iloc[:, i]

        if expected == "int":
            if not pd.api.types.is_integer_dtype(col):
                raise TypeError(
                    f"Column {i+1} is {col.dtype}. Expected int."
                )

        elif expected == "float":
            if not pd.api.types.is_float_dtype(col):
                raise TypeError(
                    f"Column {i+1} is {col.dtype}. Expected float."
                )

    # Check final column values
    if not data.iloc[:, 5].between(1, 6).all():
        print("Column 6 values must be between 1 and 6.")
        return None, False
    print(data.head())
    return data, True


In [14]:
weather_df(weather_data)

   1722  1  12  1.9  1.8  1.1
0  1722  1  13  2.3  2.2    1
1  1722  1  14  1.8  1.7    1
2  1722  1  15  0.9  0.8    1
3  1722  1  16 -1.8 -1.9    1
4  1722  1  17  0.5  0.4    1


(        1722  1  12  1.9  1.8  1.1
 0       1722  1  13  2.3  2.2    1
 1       1722  1  14  1.8  1.7    1
 2       1722  1  15  0.9  0.8    1
 3       1722  1  16 -1.8 -1.9    1
 4       1722  1  17  0.5  0.4    1
 ...      ... ..  ..  ...  ...  ...
 109921  2022  2  27  0.1 -0.2    1
 109922  2022  2  28 -4.1 -4.4    1
 109923  2022  2  29  2.8  2.6    1
 109924  2022  2  30  4.2  4.0    1
 109925  2022  2  31  5.2  4.9    1
 
 [109926 rows x 6 columns],
 True)

In [15]:
test_df = pd.DataFrame.from_dict({"Col1":[1984,1985,1986],
                                  "Col2":[4,5,6], "Col3":[10,5,26],
                                  "Col4":[10.4,12.7,22.1],
                                  "Col5":[10.1,12.9,21.4],
                                  "Col6":[1,2,1]})


In [34]:
data1, valid = weather_df(weather_data)

   1722  1  12  1.9  1.8  1.1
0  1722  1  13  2.3  2.2    1
1  1722  1  14  1.8  1.7    1
2  1722  1  15  0.9  0.8    1
3  1722  1  16 -1.8 -1.9    1
4  1722  1  17  0.5  0.4    1


In [35]:
def city_filter(df: pd.DataFrame, city: str, city_dict: dict) -> pd.DataFrame:
    """Filter rows by city using the 6th column."""

    cities = city_dict.keys()
    col_count = len(df.columns)

    if col_count != COLUMN_COUNT:
        msg = f"Expected 6 columns; Received {col_count}"
        raise InvalidColumnCountError(msg)

    if city not in city_dict:
        msg = f"{city} is not defined, defined cities are: {list(cities)}"
        raise InvalidCityError(msg)

    city_code = city_dict[city]

    return df[df.iloc[:, 5] == city_code]


In [36]:
city_filter(data1, "Uppsala", city_dict)

,1722,1,12,1.9,1.8,1.1
0,1722,1,13,2.3,2.2,1
1,1722,1,14,1.8,1.7,1
2,1722,1,15,0.9,0.8,1
3,1722,1,16,-1.8,-1.9,1
4,1722,1,17,0.5,0.4,1
...,...,...,...,...,...,...
109921,2022,2,27,0.1,-0.2,1
109922,2022,2,28,-4.1,-4.4,1
109923,2022,2,29,2.8,2.6,1
109924,2022,2,30,4.2,4.0,1


In [37]:
def yearly_average_temp(df: pd.DataFrame) -> pd.Series:
    """Calculate yearly average temp using the 4th column."""
    if "year" not in df.columns or "temp" not in df.columns:
#       Warning, year or temp not detected in df, resolving cols by order
        return df.groupby(df.iloc[:, 0]).mean().iloc[:, 3]
    return df.groupby("year").mean()["temp"]


In [40]:
data2 = city_filter(data1, "Uppsala", city_dict)
data2.head()

,1722,1,12,1.9,1.8,1.1
0,1722,1,13,2.3,2.2,1
1,1722,1,14,1.8,1.7,1
2,1722,1,15,0.9,0.8,1
3,1722,1,16,-1.8,-1.9,1
4,1722,1,17,0.5,0.4,1


In [42]:
newdf = yearly_average_temp(data2)
newdf.head()

1722
1722    7.349855
1723    8.096978
1724    5.483880
1725    5.937808
1726    6.005205
Name: 1.9, dtype: float64

In [45]:
def read_output(df):
    """Validates output dataframe format."""

    # Check input is actually a DataFrame
    if not isinstance(df, pd.DataFrame):
        print("Input is not a DataFrame")
        return None, False

    # Check number of columns
    if df.shape[1] != 2:
        print("Output must have 2 columns")
        return None, False

    # Expected column types
    expected_types = [
        "int",
        "float"
    ]

    for i, expected in enumerate(expected_types):
        col = df.iloc[:, i]

        if expected == "int":
            if not pd.api.types.is_integer_dtype(col):
                raise TypeError(
                    f"Column {i+1} is {col.dtype}. Expected int."
                )

        elif expected == "float":
            if not pd.api.types.is_float_dtype(col):
                raise TypeError(
                    f"Column {i+1} is {col.dtype}. Expected float."
                )

    return df, True




In [50]:
read_output(newdf)

Input is not a DataFrame


(None, False)

In [53]:
def write_output(df, filename="output.csv"):
    """Writes dataframe to CSV file."""

    df.to_csv(filename, index=False)

    print (f"File saved as {filename}")
    return True


In [54]:
write_output(newdf, "yearlyaveragetemp.csv")

File saved as yearlyaveragetemp.csv


True